# Deploy KYC agent to Databricks Apps

This notebook deploys the KYC onboarding agent from `agents/kyc_agent/` as a **Databricks App** exposing the Responses API (`/invocations`). Once deployed, you can chat with it via the built-in UI, AI Playground, or the CLI test script.

**Prerequisites:** Run `01_kyc_data_setup` once to create the schema, tables, and UC functions. Then run Configuration and Deploy cells below.

## 1. Install dependencies

Run once; restart kernel if needed.

In [ ]:
%pip install 'databricks-sdk>=0.38.0' mlflow pyyaml 'typing_extensions>=4.12' pg8000 --quiet
dbutils.library.restartPython()

## 2. Configuration

Defaults are set in `_shared/config`. To override, change the **catalog** and **schema** widgets at the top of the notebook after running the config cell below.

**Warehouse ID:** The deployed app uses the [SQL Statement Execution API](https://docs.databricks.com/api/workspace/statementexecution) to run queries (no Spark session). The `warehouse_id` tells it which SQL warehouse to use. Find it in **SQL Warehouses** in the workspace UI.

In [ ]:
%run ../_shared/config

In [ ]:
import os

# CATALOG, SCHEMA, CATALOG_SCHEMA, VOLUME_PATH come from %run ../_shared/config

dbutils.widgets.text("warehouse_id", "", "Warehouse ID")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
DEMO_SBI_CODE = "47.11"  # 47.11 = low risk (SME opened), 64.11 = high risk (SME blocked)

if not WAREHOUSE_ID:
    raise ValueError("warehouse_id is required. Set it in the widget at the top of the notebook. "
                     "Find it in SQL Warehouses in the workspace UI (the hex string in the URL or details panel).")

print(f"CATALOG_SCHEMA={CATALOG_SCHEMA}, VOLUME_PATH={VOLUME_PATH}, DEMO_SBI_CODE={DEMO_SBI_CODE}, WAREHOUSE_ID={WAREHOUSE_ID}")

### Run setup

Creates the schema, tables, and UC functions in `01_kyc_data_setup`. Skips if they already exist.

In [ ]:
%run ./01_kyc_data_setup

## 3. Deploy KYC agent to Databricks Apps

Deploy the KYC onboarding agent from **agents/kyc_agent/** so it exposes the Responses API. The orchestrator will then call it as `apps/<KYC_APP_NAME>`.

**Option A – Run deploy from this notebook:** Set `KYC_APP_NAME` and ensure `agents/kyc_agent` is available (synced via `databricks bundle deploy`). Run the cell below.

**Option B – Run deploy from your local machine:** From the repo root, deploy manually, then set `KYC_APP_NAME` in the next cell and skip the deploy cell.

In [ ]:
# App name used when deploying and when calling the app as subagent (model = apps/<name>)
KYC_APP_NAME = "kyc-onboarding-agent"
LAKEBASE_INSTANCE = "kyc-lakebase"

# Resolve workspace path to agents/kyc_agent (parent of notebooks/scene1_kyc_onboarding/<name>)
import os
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_repo_root = "/".join(_nb_path.rstrip("/").split("/")[:-3])  # strip /notebooks/scene1_kyc_onboarding/<name>
APP_SOURCE_PATH = "/Workspace" + _repo_root + "/agents/kyc_agent"  # full workspace filesystem path
print(f"KYC_APP_NAME={KYC_APP_NAME}, LAKEBASE_INSTANCE={LAKEBASE_INSTANCE}, APP_SOURCE_PATH={APP_SOURCE_PATH}")

Create an MLflow experiment for the KYC app so deployed app traces have a destination (fixes "experiment_id is missing" in app logs). The deploy cell below will inject this ID into the app's `app.yaml` before deploying.

In [ ]:
# Create or get MLflow experiment for KYC app traces (used when we patch app.yaml before deploy).
import mlflow

mlflow.set_tracking_uri("databricks")
_current_user = spark.sql("SELECT current_user()").first()[0]
KYC_APP_EXPERIMENT_NAME = f"/Users/{_current_user}/kyc_agent_app_experiment"

existing = mlflow.get_experiment_by_name(KYC_APP_EXPERIMENT_NAME)
if existing:
    KYC_APP_MLFLOW_EXPERIMENT_ID = existing.experiment_id
    print(f"Using existing experiment: {KYC_APP_EXPERIMENT_NAME} -> {KYC_APP_MLFLOW_EXPERIMENT_ID}")
else:
    KYC_APP_MLFLOW_EXPERIMENT_ID = mlflow.create_experiment(KYC_APP_EXPERIMENT_NAME)
    print(f"Created experiment: {KYC_APP_EXPERIMENT_NAME} -> {KYC_APP_MLFLOW_EXPERIMENT_ID}")

In [ ]:
# Deploy the KYC agent app (create app if needed, deploy from workspace path).
# Patches app.yaml with MLFLOW_EXPERIMENT_ID, WAREHOUSE_ID, CATALOG_SCHEMA, VOLUME_PATH, DEMO_SBI_CODE, LAKEBASE_INSTANCE_NAME.
import os
import yaml
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App, AppDeployment

w = WorkspaceClient()

if not os.path.isdir(APP_SOURCE_PATH):
    print(f"agents/kyc_agent not found at {APP_SOURCE_PATH}.")
    print("Run 'databricks bundle deploy' first to sync agents/kyc_agent to the workspace.")
else:
    # Patch app.yaml with current widget values
    app_yaml_path = os.path.join(APP_SOURCE_PATH, "app.yaml")
    if os.path.isfile(app_yaml_path):
        with open(app_yaml_path, "r") as f:
            cfg = yaml.safe_load(f)
        env = cfg.get("env") or []

        patches = {
            "WAREHOUSE_ID": WAREHOUSE_ID,
            "CATALOG_SCHEMA": CATALOG_SCHEMA,
            "VOLUME_PATH": VOLUME_PATH,
            "DEMO_SBI_CODE": DEMO_SBI_CODE,
            "LAKEBASE_INSTANCE_NAME": LAKEBASE_INSTANCE,
        }
        if "KYC_APP_MLFLOW_EXPERIMENT_ID" in dir():
            patches["MLFLOW_EXPERIMENT_ID"] = str(KYC_APP_MLFLOW_EXPERIMENT_ID)

        for key, value in patches.items():
            env = [e for e in env if e.get("name") != key]
            env.append({"name": key, "value": value})

        cfg["env"] = env
        with open(app_yaml_path, "w") as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
        print(f"Patched app.yaml with: {', '.join(patches.keys())}")

    # Create app (skip if already exists)
    try:
        w.apps.get(KYC_APP_NAME)
        print(f"App {KYC_APP_NAME} already exists.")
    except Exception:
        print(f"Creating app {KYC_APP_NAME}...")
        w.apps.create_and_wait(app=App(name=KYC_APP_NAME))
        print(f"App {KYC_APP_NAME} created.")

    # Deploy
    print(f"Deploying {KYC_APP_NAME} from {APP_SOURCE_PATH}...")
    deployment = w.apps.deploy_and_wait(
        app_name=KYC_APP_NAME,
        app_deployment=AppDeployment(source_code_path=APP_SOURCE_PATH),
    )
    print(f"Deployed {KYC_APP_NAME} (status: {deployment.status.state.value}). Orchestrator will call it as apps/{KYC_APP_NAME}")

### Create Lakebase instance (if needed)

The agent uses [Lakebase](https://docs.databricks.com/aws/en/lakebase/) (managed PostgreSQL) for conversation memory via LangGraph checkpointing. The cell below creates a `kyc-lakebase` instance if one doesn't already exist. This takes 2–5 minutes on first run.

In [ ]:
# Create Lakebase instance if it doesn't already exist.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance

w = WorkspaceClient()

try:
    info = w.database.get_database_instance(LAKEBASE_INSTANCE)
    print(f"Lakebase instance '{LAKEBASE_INSTANCE}' already exists (state: {info.state.value}).")
except Exception:
    print(f"Creating Lakebase instance '{LAKEBASE_INSTANCE}' (this takes 2–5 minutes)...")
    info = w.database.create_database_instance_and_wait(
        DatabaseInstance(name=LAKEBASE_INSTANCE, capacity="CU_1")
    )
    print(f"Lakebase instance '{LAKEBASE_INSTANCE}' created (state: {info.state.value}).")

### Grant permissions to the app service principal

The deployed app runs as its own service principal (SP). It needs:
1. **Unity Catalog** — `USE SCHEMA`, `SELECT`, `EXECUTE`, `MODIFY`, `READ VOLUME` so it can call UC functions, parse documents, and write to the accounts table.
2. **SQL Warehouse** — `CAN_USE` so it can execute SQL via the Statement Execution API.
3. **MLflow** — `CAN_MANAGE` on the tracing experiment so the app can export traces.
4. **Lakebase** — resource binding + PostgreSQL grants so the checkpointer can create and read checkpoint tables for conversation memory.

In [ ]:
# Grant the app's service principal access to the catalog, schema, warehouse, MLflow experiment, and Lakebase.
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import App, AppResource, AppResourceDatabase, AppResourceDatabaseDatabasePermission, AppDeployment
from databricks.sdk.service.iam import AccessControlRequest, PermissionLevel

w = WorkspaceClient()
app_info = w.apps.get(KYC_APP_NAME)
sp_id = app_info.service_principal_client_id

# 1. Unity Catalog grants — USE SCHEMA, SELECT, EXECUTE, MODIFY, READ VOLUME (catalog-level grant skipped; SP must already have USE CATALOG)
uc_grants = [
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG_SCHEMA} TO `{sp_id}`",
    f"GRANT SELECT ON SCHEMA {CATALOG_SCHEMA} TO `{sp_id}`",
    f"GRANT EXECUTE ON SCHEMA {CATALOG_SCHEMA} TO `{sp_id}`",
    f"GRANT MODIFY ON SCHEMA {CATALOG_SCHEMA} TO `{sp_id}`",
    f"GRANT READ VOLUME ON SCHEMA {CATALOG_SCHEMA} TO `{sp_id}`",
]
for stmt in uc_grants:
    spark.sql(stmt)
print(f"1/6 Granted UC permissions to app SP {sp_id}")

# 2. SQL Warehouse — CAN_USE so the app can execute SQL via Statement Execution API
try:
    w.permissions.update(
        request_object_type="sql/warehouses",
        request_object_id=WAREHOUSE_ID,
        access_control_list=[
            AccessControlRequest(
                service_principal_name=sp_id,
                permission_level=PermissionLevel.CAN_USE,
            ),
        ],
    )
    print(f"2/6 Granted CAN_USE on SQL warehouse {WAREHOUSE_ID} to app SP {sp_id}")
except Exception as e:
    print(f"2/6 SQL warehouse permission skipped: {e}")

# 3. MLflow experiment — CAN_MANAGE so the app can export traces
# Use update() (not set()) to add the SP permission without overwriting existing permissions
try:
    w.permissions.update(
        request_object_type="experiments",
        request_object_id=str(KYC_APP_MLFLOW_EXPERIMENT_ID),
        access_control_list=[
            AccessControlRequest(
                service_principal_name=sp_id,
                permission_level=PermissionLevel.CAN_MANAGE,
            ),
        ],
    )
    print(f"3/6 Granted CAN_MANAGE on MLflow experiment {KYC_APP_MLFLOW_EXPERIMENT_ID} to app SP {sp_id}")
except Exception as e:
    print(f"3/6 MLflow experiment permission skipped: {e}")

# 4. Lakebase — bind database resource so the app can use the checkpointer
try:
    w.apps.update(
        KYC_APP_NAME,
        App(name=KYC_APP_NAME, resources=[AppResource(
            name="database",
            database=AppResourceDatabase(
                instance_name=LAKEBASE_INSTANCE,
                database_name="databricks_postgres",
                permission=AppResourceDatabaseDatabasePermission.CAN_CONNECT_AND_CREATE,
            ),
        )]),
    )
    print(f"4/6 Bound Lakebase resource ({LAKEBASE_INSTANCE}) to {KYC_APP_NAME}")
except Exception as e:
    print(f"4/6 Lakebase binding skipped: {e}")

# 5. Redeploy app so the Lakebase binding takes effect and the PostgreSQL role is created
print(f"5/6 Redeploying {KYC_APP_NAME} to provision PostgreSQL role...")
deployment = w.apps.deploy_and_wait(
    app_name=KYC_APP_NAME,
    app_deployment=AppDeployment(source_code_path=APP_SOURCE_PATH),
)
print(f"5/6 App redeployed (status: {deployment.status.state.value})")

# 6. PostgreSQL grants — retry up to 5 times (role needs time to propagate after deploy)
import ssl
import pg8000

pg_ok = False
for attempt in range(1, 6):
    try:
        _cred = w.database.generate_database_credential(instance_names=[LAKEBASE_INSTANCE], request_id=f"nb-{attempt}")
        _inst = w.database.get_database_instance(LAKEBASE_INSTANCE)
        _user = spark.sql("SELECT current_user()").first()[0]
        ssl_context = ssl.create_default_context()
        conn = pg8000.connect(
            host=_inst.read_write_dns, port=5432, database="databricks_postgres",
            user=_user, password=_cred.token, ssl_context=ssl_context,
            timeout=10,
        )
        conn.autocommit = True
        cur = conn.cursor()
        for s in [
            f'GRANT ALL ON SCHEMA public TO "{sp_id}"',
            f'GRANT ALL ON ALL TABLES IN SCHEMA public TO "{sp_id}"',
            f'GRANT ALL ON ALL SEQUENCES IN SCHEMA public TO "{sp_id}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT ALL ON TABLES TO "{sp_id}"',
            f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT ALL ON SEQUENCES TO "{sp_id}"',
        ]:
            cur.execute(s)
        conn.close()
        print(f"6/6 Granted PostgreSQL permissions to app SP {sp_id} on {LAKEBASE_INSTANCE}")
        pg_ok = True
        break
    except Exception as e:
        if attempt < 5:
            print(f"6/6 PostgreSQL attempt {attempt}/5 failed ({e}), retrying in 15s...")
            time.sleep(15)
        else:
            print(f"6/6 PostgreSQL grants failed after 5 attempts: {e}")

if pg_ok:
    print(f"\nAll permissions granted. App is ready at: {w.apps.get(KYC_APP_NAME).url}")
else:
    print(f"\nAll permissions granted except PostgreSQL. Run the cell below to retry manually.")

---

## Delete resources (optional)

Run the cell below to clean up all resources created by this demo: the Databricks App, Lakebase instance, tables, UC functions, volume, and schema. Each resource is checked before deletion — if it doesn't exist, it is skipped.

In [ ]:
# Clean up all demo resources. Each step is wrapped in try/except so partial cleanup works.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
_cat, _sch = CATALOG_SCHEMA.split(".", 1)

# 1. Delete the Databricks App
try:
    w.apps.delete(KYC_APP_NAME)
    print(f"Deleted app: {KYC_APP_NAME}")
except Exception as e:
    print(f"App delete skipped: {e}")

# 2. Delete the Lakebase instance
try:
    w.database.delete_database_instance(LAKEBASE_INSTANCE)
    print(f"Deleted Lakebase instance: {LAKEBASE_INSTANCE}")
except Exception as e:
    print(f"Lakebase delete skipped: {e}")

# 3. Delete the MLflow experiment
try:
    import mlflow
    mlflow.set_tracking_uri("databricks")
    exp = mlflow.get_experiment_by_name(KYC_APP_EXPERIMENT_NAME)
    if exp:
        mlflow.delete_experiment(exp.experiment_id)
        print(f"Deleted MLflow experiment: {KYC_APP_EXPERIMENT_NAME}")
    else:
        print("MLflow experiment not found, skipping.")
except Exception as e:
    print(f"MLflow experiment delete skipped: {e}")

# 4. Drop UC functions
for fn in ["check_pep", "check_sanctions", "check_sanctioned_country", "check_customer_risk", "check_fraud"]:
    try:
        spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG_SCHEMA}.{fn}")
    except Exception:
        pass
print("Dropped UC functions.")

# 5. Drop tables
for tbl in ["pep_screening", "sanctions_screening", "sanctioned_countries", "fraud_list", "accounts", "sbi_risk_codes"]:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG_SCHEMA}.{tbl}")
    except Exception:
        pass
print("Dropped tables.")

# 6. Drop volume
try:
    spark.sql(f"DROP VOLUME IF EXISTS {_cat}.{_sch}.agent_volume")
    print("Dropped volume.")
except Exception as e:
    print(f"Volume drop skipped: {e}")

# 7. Drop schema
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_SCHEMA}")
    print(f"Dropped schema: {CATALOG_SCHEMA}")
except Exception as e:
    print(f"Schema drop skipped: {e}")

print("Cleanup complete.")